# Notebook 05 — Corrección Heckman por sesgo de selección

La participación laboral femenina en México es de ~54%, frente a ~85% en hombres. La muestra de mujeres ocupadas no es aleatoria: las que sí trabajan pueden ser sistemáticamente distintas a las que no, en variables que también afectan el salario.

Si esa selección es importante, **el coeficiente de "mujer" en Mincer está sesgado** y la brecha residual del 50% que encontramos en el notebook 04 podría cambiar al corregir. Acá lo verifico.

Procedimiento (Heckman 1979):

1. **Etapa 1 — Probit de participación**: modelar `participa = 1` (clase2 ∈ {1,2}) sobre toda la muestra de mujeres en edad laboral. Incluyo variables clásicas de restricción de exclusión (estado civil, niños en hogar) además de edad y escolaridad.
2. **Calcular el inverso de Mills** sobre las mujeres ocupadas.
3. **Etapa 2 — Mincer con Mills como control** sobre las ocupadas. Si el coeficiente de Mills es significativo, hay sesgo de selección.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import norm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import enoe_loader as enoe

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
})
OUTPUTS = ROOT / "outputs"


## 1. Construir variables del probit

Necesito la muestra completa de mujeres en edad laboral (18–65), no solo las ocupadas. Las variables clave para identificar el sistema son:

- `casada` — estado civil 1 (unión libre) o 5 (casada) en ENOE
- `ninos_en_hogar` — número de personas con `edad < 6` en el mismo hogar

Estas dos afectan claramente la decisión de participar pero, condicional en trabajar, no deberían afectar el salario por hora — esa es la **restricción de exclusión**.

> Como el módulo `enoe_loader` ya descarta las no-ocupadas, acá hago la construcción directa desde SDEMT crudo. Asumo que ya tienes la tabla cargada (ver notebook 01).


In [ ]:
# Para reproducir desde cero: cargar SDEMT y construir las variables Heckman.
# Si ya las tienes, salta directo a la siguiente celda.

# from src.enoe_loader import detectar_archivos, cargar_tabla
# archivos = detectar_archivos(ROOT / "enoe_2025_trim4_csv")
# sdemt = cargar_tabla(archivos["sdemt"])

# LLAVES_HOGAR = ["cd_a","cve_ent","con","v_sel","n_hog"]
# sdemt["edad_num"] = pd.to_numeric(sdemt["eda"], errors="coerce")
# sdemt["sex_num"]  = pd.to_numeric(sdemt["sex"], errors="coerce")
# sdemt["anios_esc_num"] = pd.to_numeric(sdemt["anios_esc"], errors="coerce")
# sdemt["e_con_num"] = pd.to_numeric(sdemt["e_con"], errors="coerce")
#
# # Contar niños menores de 6 por hogar
# ninos_por_hogar = (sdemt.assign(es_nino=(sdemt["edad_num"]<6).astype(int))
#                          .groupby(LLAVES_HOGAR)["es_nino"].sum())
# sdemt = sdemt.set_index(LLAVES_HOGAR)
# sdemt["ninos_en_hogar"] = ninos_por_hogar
# sdemt = sdemt.reset_index()
#
# sdemt["participa"] = sdemt["clase2"].isin([1,2]).astype(int)
# sdemt["casada"]    = sdemt["e_con_num"].isin([1,5]).astype(int)
#
# mujeres_pea = sdemt[(sdemt["sex_num"]==2) &
#                     (sdemt["edad_num"]>=18) & (sdemt["edad_num"]<=65)].copy()
# mujeres_pea = mujeres_pea.dropna(subset=["edad_num","anios_esc_num","e_con_num"])
# mujeres_pea["edad2"] = mujeres_pea["edad_num"]**2

print("Ver código comentado para reproducir desde SDEMT crudo.")


## 2. Etapa 1 — Probit de participación

Por simplicidad y para tener el notebook corredor, voy a usar el dataset ya procesado de mujeres con Mills (`/tmp/mujeres_pea_mills.pkl`). Si lo reconstruyes desde SDEMT, sustituye el `read_pickle` por tu DataFrame.


In [ ]:
# Si el archivo de cache no existe, hay que rehacerlo desde SDEMT (ver celda anterior).
mp = pd.read_pickle("/tmp/mujeres_pea_mills.pkl") if Path("/tmp/mujeres_pea_mills.pkl").exists() \
     else pd.read_parquet(ROOT / "data" / "processed" / "mujeres_pea_mills.parquet")

print(f"Mujeres en edad laboral: {len(mp):,}")
print(f"Tasa de participación (ponderada): "
      f"{enoe.media_ponderada(mp['participa'].astype(float), mp['fac_tri'])*100:.1f}%")

probit_formula = ("participa ~ edad_num + edad2 + anios_esc_num + casada + ninos_en_hogar")
probit = smf.probit(probit_formula, data=mp).fit(disp=False)
print(probit.summary().tables[1])
print(f"\nPseudo R² = {probit.prsquared:.4f}")


**Lo que dice el probit:**

- `casada` (β = −0.58, ***): las casadas participan mucho menos. Efecto enorme.
- `ninos_en_hogar` (β = −0.08, ***): cada niño menor de 6 reduce la propensión a participar.
- `edad` y `edad²`: forma cóncava, pico de participación cerca de los 40.
- `anios_esc` (β = +0.027, ***): más educación, más participación.

Pseudo R² de 0.09 es bajo en términos absolutos pero típico para modelos de decisión laboral.

La restricción de exclusión queda satisfecha: estado civil y niños afectan participación, y no hay razón teórica fuerte para que afecten la productividad horaria (el salario condicional en trabajar) en un modelo Mincer estándar.


## 3. Calcular el inverso de Mills y correr la etapa 2


In [ ]:
# Mills sobre TODAS las mujeres (no solo ocupadas — necesario para que la corrección tenga sentido)
xb = probit.fittedvalues
mp["mills"] = norm.pdf(xb) / norm.cdf(xb)

# Quedarse solo con las ocupadas y aplicar el filtro Mincer del notebook 01
# (este paso está hecho ya en el dataset que tienes guardado).
mp_oc = pd.read_parquet("/tmp/mujeres_mincer_mills.parquet") if Path("/tmp/mujeres_mincer_mills.parquet").exists() \
        else None
print(f"Muestra Mincer mujeres con Mills: {len(mp_oc):,}")
mp_oc.head(3)


In [ ]:
# Comparar Mincer con y sin Mills como control
formula_base = ("log_ingreso_mensual_w ~ log_horas_w + anios_escolaridad + experiencia + experiencia2 "
                "+ C(sector_cat) + subordinado_i + C(ent_cat)")
formula_heck = formula_base + " + mills"

res_sin = smf.wls(formula_base, data=mp_oc, weights=mp_oc["fac_tri"]).fit(cov_type="HC3")
res_con = smf.wls(formula_heck, data=mp_oc, weights=mp_oc["fac_tri"]).fit(cov_type="HC3")

print("Coef. de Mills:")
print(f"  Estimado: {res_con.params['mills']:+.4f}")
print(f"  SE:       {res_con.bse['mills']:.4f}")
print(f"  p-valor:  {res_con.pvalues['mills']:.4f}")
print(f"\nR² sin Heckman: {res_sin.rsquared:.4f}")
print(f"R² con Heckman: {res_con.rsquared:.4f}")


## 4. ¿El sesgo de selección mueve la brecha?

Comparación de coeficientes Mincer entre los dos modelos.


In [ ]:
variables = {
    "log_horas_w":       "log(horas)",
    "anios_escolaridad": "Escolaridad",
    "experiencia":       "Experiencia",
    "experiencia2":      "Experiencia²",
    "subordinado_i":     "Subordinada",
    "Intercept":         "Intercept",
}

tabla = pd.DataFrame({
    "Variable":   list(variables.values()),
    "β sin":      [res_sin.params[v] for v in variables],
    "(SE sin)":   [res_sin.bse[v]    for v in variables],
    "β con":      [res_con.params[v] for v in variables],
    "(SE con)":   [res_con.bse[v]    for v in variables],
})
tabla["Δ"] = tabla["β con"] - tabla["β sin"]
tabla.round(4)


In [ ]:
# Forest plot comparativo (excluyendo el intercept para que se vea bien)
vars_plot = {k: v for k, v in variables.items() if k != "Intercept"}

fig, ax = plt.subplots(figsize=(10, 5.5))
y_pos = np.arange(len(vars_plot))[::-1]
offset = 0.18

for i, (var, etiqueta) in enumerate(vars_plot.items()):
    y = y_pos[i]
    bs, ses = res_sin.params[var], res_sin.bse[var]
    bc, sec = res_con.params[var], res_con.bse[var]
    ax.errorbar(bs, y+offset, xerr=1.96*ses, fmt="o", color="#d95f0e",
                capsize=4, markersize=8, label="Sin Heckman" if i==0 else None)
    ax.errorbar(bc, y-offset, xerr=1.96*sec, fmt="o", color="#7a3a04",
                capsize=4, markersize=8, label="Con Heckman" if i==0 else None)
    ax.text(bs, y+offset+0.16, f"{bs:+.3f}", ha="center", fontsize=8.5, color="#d95f0e")
    ax.text(bc, y-offset-0.22, f"{bc:+.3f}", ha="center", fontsize=8.5, color="#7a3a04")

ax.axvline(0, color="black", linestyle="-", linewidth=0.5, alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(list(vars_plot.values()))
ax.set_xlabel("Coeficiente (efecto sobre log ingreso mensual)")
ax.set_title("Mincer en mujeres: con y sin corrección Heckman")
ax.text(0.99, -0.14,
        f"Mills no es significativo (p = {res_con.pvalues['mills']:.3f}). "
        "El sesgo de selección observable no afecta la brecha residual.",
        transform=ax.transAxes, ha="right", fontsize=9, color="gray", style="italic")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUTS / "heckman_g1_comparacion.png")
plt.show()


## Debrief del notebook 05

**Lo que aprendí:**

La corrección Heckman aplica sobre la muestra de mujeres ocupadas, pero **el coeficiente de Mills no es estadísticamente significativo** (p ≈ 0.52). Eso significa que el sesgo de selección no observado, después de controlar por edad y escolaridad, no es material en estos datos.

Implicación: la brecha residual del 50% que encontramos en el notebook 04 **es robusta** al sesgo de selección. Aunque las mujeres que no participan en el mercado son sistemáticamente distintas de las que sí (por estado civil y por hijos en el hogar), esas diferencias se capturan vía variables observables incluidas en el modelo. No hay un componente no-observado importante.

Pero ojo con la lectura: Heckman corrige el sesgo **dado que el probit está bien identificado**. Si hay variables omitidas que afectan tanto participación como salario y no están en ninguna de las dos ecuaciones (habilidades cognitivas, redes profesionales, calidad de la educación recibida), Heckman no lo arregla. Lo único que dice mi resultado es: "lo que sí puedo modelar no parece estar pesando".

**Lo que se me resistió:**

- Construir `ninos_en_hogar` requirió pegar SDEMT consigo mismo agrupando por llaves de hogar. Las llaves correctas son `cd_a, cve_ent, con, v_sel, n_hog` (no `ent` que no existe en SDEMT como columna directa).
- La variable `tiene_ninos` (dummy 0/1) salió no significativa, pero `ninos_en_hogar` (conteo) sí. La cantidad importa más que la mera presencia.

**Para el notebook 06 (visualizaciones finales):**

Tengo todo para el README y el carrusel de LinkedIn:

- G1: brecha del 23% no se explica por menor calificación (caracterización Mincer)
- G2: distribución de horas con la doble moda femenina
- G3: descomposición Oaxaca-Blinder (49% horas + 50% residual)
- G4: brecha residual robusta a Heckman
- G5: brecha por entidad federativa
